In [2]:
import os, time, math, random
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from PIL import Image
import torchvision.transforms.functional as TF

In [ ]:
# -------------------------
# Dataset 
# -------------------------
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff")

class RandomPatchSigmaMapDataset(Dataset):
    """
    Random patches from clean images + AWGN with random sigma.
    Returns:
      inp   (4,H,W) = noisy_rgb (3) + sigma_map (1)
      clean (3,H,W)
    """
    def __init__(self, clean_dir: str, patch: int = 128, sigma_min: float = 0.0, sigma_max: float = 50.0):
        self.paths = []
        for root, _, files in os.walk(clean_dir):
            for f in files:
                if f.lower().endswith(IMG_EXT):
                    self.paths.append(os.path.join(root, f))
        if not self.paths:
            raise ValueError(f"No images found in {clean_dir}")

        self.patch = patch
        self.sigma_min = float(sigma_min)
        self.sigma_max = float(sigma_max)

    def __len__(self):
        #longueur “virtuelle”
        return 1_000_000

    def _random_crop(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        p = self.patch
        if w < p or h < p:
            scale = max(p / w, p / h)
            nw, nh = int(round(w * scale)), int(round(h * scale))
            img = img.resize((nw, nh), Image.BICUBIC)
            w, h = img.size

        x0 = random.randint(0, w - p)
        y0 = random.randint(0, h - p)
        return img.crop((x0, y0, x0 + p, y0 + p))

    def _augment(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.5:
            img = TF.hflip(img)
        if random.random() < 0.5:
            img = TF.vflip(img)
        k = random.randint(0, 3)
        if k:
            img = img.rotate(90 * k)
        return img

    def __getitem__(self, idx):
        path = random.choice(self.paths)
        img = Image.open(path).convert("RGB")
        img = self._random_crop(img)
        img = self._augment(img)

        clean = TF.to_tensor(img)  # (3,H,W) in [0,1]

        # sigma en "pixel space" 0..50 (comme le papier)
        sigma = random.uniform(self.sigma_min, self.sigma_max)

        # IMPORTANT: PAS DE CLIP DU NOISY (papier)
        noise = torch.randn_like(clean) * (sigma / 255.0)
        noisy = clean + noise  # peut sortir de [0,1]

        sigma_map = torch.full((1, clean.shape[1], clean.shape[2]), sigma / 255.0)
        inp = torch.cat([noisy, sigma_map], dim=0)  # (4,H,W)
        return inp, clean



In [4]:
# -------------------------
# DRUNet (papier): bias-free, 4 scales, SConv 2x2, TConv 2x2, nb=4
# -------------------------
class ResBlockOneReLU(nn.Module):
    """
    Residual block: Conv -> ReLU -> Conv, bias-free, un seul ReLU.
    """
    def __init__(self, nc: int):
        super().__init__()
        self.c1 = nn.Conv2d(nc, nc, 3, 1, 1, bias=False)
        self.act = nn.ReLU(inplace=True)
        self.c2 = nn.Conv2d(nc, nc, 3, 1, 1, bias=False)

    def forward(self, x):
        y = self.c1(x)
        y = self.act(y)
        y = self.c2(y)
        return x + y


class SConv2x2(nn.Module):
    """2×2 strided conv downscale (pas d'activation après, papier)"""
    def __init__(self, in_nc: int, out_nc: int):
        super().__init__()
        self.conv = nn.Conv2d(in_nc, out_nc, kernel_size=2, stride=2, padding=0, bias=False)

    def forward(self, x):
        return self.conv(x)


class TConv2x2(nn.Module):
    """2×2 transposed conv upscale (pas d'activation après, papier)"""
    def __init__(self, in_nc: int, out_nc: int):
        super().__init__()
        self.tconv = nn.ConvTranspose2d(in_nc, out_nc, kernel_size=2, stride=2, padding=0, bias=False)

    def forward(self, x):
        return self.tconv(x)


def _pad_to_multiple(x: torch.Tensor, mult: int = 8) -> Tuple[torch.Tensor, Tuple[int,int,int,int]]:
    _, _, h, w = x.shape
    pad_h = (mult - h % mult) % mult
    pad_w = (mult - w % mult) % mult
    pt = pad_h // 2
    pb = pad_h - pt
    pl = pad_w // 2
    pr = pad_w - pl
    if pad_h or pad_w:
        x = F.pad(x, (pl, pr, pt, pb), mode="reflect")
    return x, (pl, pr, pt, pb)

def _unpad(x: torch.Tensor, pads: Tuple[int,int,int,int]) -> torch.Tensor:
    pl, pr, pt, pb = pads
    if (pl, pr, pt, pb) == (0,0,0,0):
        return x
    return x[:, :, pt:x.shape[2]-pb, pl:x.shape[3]-pr]


class DRUNetSigmaMap(nn.Module):
    """
    Input:  (B,4,H,W) noisy RGB + sigma_map
    Output: (B,3,H,W) denoised
    Conforme au papier: 4 scales, nc=[64,128,256,512], nb=4,
    bias-free, SConv 2x2, TConv 2x2, pas d'activation après head/tail/SConv/TConv. :contentReference[oaicite:4]{index=4}
    """
    def __init__(self, in_nc=4, out_nc=3, nc=(64,128,256,512), nb=4):
        super().__init__()
        c1, c2, c3, c4 = nc

        # head: pas d'activation après
        self.head = nn.Conv2d(in_nc, c1, 3, 1, 1, bias=False)

        # encoder
        self.e1 = nn.Sequential(*[ResBlockOneReLU(c1) for _ in range(nb)])
        self.d1 = SConv2x2(c1, c2)

        self.e2 = nn.Sequential(*[ResBlockOneReLU(c2) for _ in range(nb)])
        self.d2 = SConv2x2(c2, c3)

        self.e3 = nn.Sequential(*[ResBlockOneReLU(c3) for _ in range(nb)])
        self.d3 = SConv2x2(c3, c4)

        # bottleneck
        self.mid = nn.Sequential(*[ResBlockOneReLU(c4) for _ in range(nb)])

        # decoder
        self.u3 = TConv2x2(c4, c3)
        self.f3 = nn.Conv2d(c3 + c3, c3, 3, 1, 1, bias=False)
        self.p3 = nn.Sequential(*[ResBlockOneReLU(c3) for _ in range(nb)])

        self.u2 = TConv2x2(c3, c2)
        self.f2 = nn.Conv2d(c2 + c2, c2, 3, 1, 1, bias=False)
        self.p2 = nn.Sequential(*[ResBlockOneReLU(c2) for _ in range(nb)])

        self.u1 = TConv2x2(c2, c1)
        self.f1 = nn.Conv2d(c1 + c1, c1, 3, 1, 1, bias=False)
        self.p1 = nn.Sequential(*[ResBlockOneReLU(c1) for _ in range(nb)])

        # tail: pas d'activation après
        self.tail = nn.Conv2d(c1, out_nc, 3, 1, 1, bias=False)

    def forward(self, inp):
        # pad pour tailles quelconques (facteur 8 car 3 downsamples)
        x, pads = _pad_to_multiple(inp, mult=8)

        x1 = self.e1(self.head(x))
        x2 = self.e2(self.d1(x1))
        x3 = self.e3(self.d2(x2))
        x4 = self.mid(self.d3(x3))

        y3 = self.u3(x4)
        y3 = self.p3(self.f3(torch.cat([y3, x3], dim=1)))

        y2 = self.u2(y3)
        y2 = self.p2(self.f2(torch.cat([y2, x2], dim=1)))

        y1 = self.u1(y2)
        y1 = self.p1(self.f1(torch.cat([y1, x1], dim=1)))

        out = self.tail(y1)
        out = _unpad(out, pads)
        return out



In [5]:

# -------------------------
# Train (même style que IRCNN)
# -------------------------
def train_drunet(
    clean_dir=r"./BSDS300/images/train",
    out_dir="weights_drunet_sigmap",
    patch=128,
    sigma_min=0.0,
    sigma_max=50.0,
    batch_size=4,
    steps_per_epoch=1000,
    max_epochs=10,
    lr0=1e-4,
    lr1=5e-5,
    plateau_epochs=5,
    log_every=50,
    num_workers=0,
    use_amp=False,         
    grad_clip=1.0
):
    os.makedirs(out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    ds = RandomPatchSigmaMapDataset(clean_dir, patch=patch, sigma_min=sigma_min, sigma_max=sigma_max)
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
        drop_last=True
    )

    model = DRUNetSigmaMap(in_nc=4, out_nc=3, nc=(64,128,256,512), nb=4).to(device).train()
    opt = Adam(model.parameters(), lr=lr0)
    loss_fn = nn.L1Loss()  # papier: L1 :contentReference[oaicite:5]{index=5}

    # AMP (optionnel)
    if device == "cuda":
        from torch.cuda.amp import GradScaler, autocast
        scaler = GradScaler(enabled=use_amp)
        autocast_ctx = lambda: autocast(enabled=use_amp)
    else:
        scaler = None
        autocast_ctx = lambda: torch.no_grad()  # dummy, pas utilisé

    best = float("inf")
    stagnant = 0
    using_lr1 = False

    for epoch in range(1, max_epochs + 1):
        running = 0.0
        start_t = time.time()

        pbar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch}/{max_epochs}", leave=True)
        it = iter(dl)

        for step in range(steps_per_epoch):
            try:
                inp, clean = next(it)
            except StopIteration:
                it = iter(dl)
                inp, clean = next(it)

            inp = inp.to(device, non_blocking=True)
            clean = clean.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            if device == "cuda":
                with autocast_ctx():
                    pred = model(inp)
                    loss = loss_fn(pred, clean)
                if not torch.isfinite(loss):
                    print("[WARN] loss NaN/Inf, skip")
                    continue
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                if grad_clip is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(opt)
                scaler.update()
            else:
                pred = model(inp)
                loss = loss_fn(pred, clean)
                if not torch.isfinite(loss):
                    print("[WARN] loss NaN/Inf, skip")
                    continue
                loss.backward()
                if grad_clip is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                opt.step()

            running += float(loss.item())

            if (step + 1) % log_every == 0:
                avg_so_far = running / (step + 1)
                elapsed = time.time() - start_t
                it_s = (step + 1) / max(elapsed, 1e-9)
                pbar.set_postfix({
                    "loss": f"{avg_so_far:.5f}",
                    "lr": f"{opt.param_groups[0]['lr']:.1e}",
                    "it/s": f"{it_s:.2f}"
                })

            pbar.update(1)

        pbar.close()
        avg = running / steps_per_epoch

        ckpt = os.path.join(out_dir, f"drunet_sigmap_epoch{epoch:02d}.pth")
        torch.save({"model": model.state_dict(), "epoch": epoch}, ckpt)
        print(f"Epoch {epoch:02d} done | avg_loss={avg:.6f} | lr={opt.param_groups[0]['lr']:.1e}")

        # LR schedule
        if avg < best - 1e-7:
            best = avg
            stagnant = 0
        else:
            stagnant += 1

        if (not using_lr1) and stagnant >= plateau_epochs:
            for g in opt.param_groups:
                g["lr"] = lr1
            using_lr1 = True
            stagnant = 0
            print(f"Switch LR to {lr1}")

        if using_lr1 and stagnant >= plateau_epochs:
            print("Early stop: loss plateaued.")
            break

    final_path = os.path.join(out_dir, "drunet_sigmap_final.pth")
    torch.save({"model": model.state_dict()}, final_path)
    print(f"Saved: {final_path}")
    return final_path


In [6]:
# -------------------------
# Utils test (comme IRCNN)
# -------------------------
def psnr_torch(x, y, eps=1e-8):
    mse = torch.mean((x - y) ** 2).item()
    return 10.0 * math.log10(1.0 / (mse + eps))

@torch.no_grad()
def denoise_image_pil(model, img_pil, sigma, device):
    """
    sigma: noise level en 'pixel space' (0..50)
    """
    y = TF.to_tensor(img_pil)  # [0,1]
    sigma_map = torch.full((1, y.shape[1], y.shape[2]), sigma / 255.0)
    inp = torch.cat([y, sigma_map], dim=0).unsqueeze(0).to(device)
    out = model(inp).squeeze(0).clamp(0.0, 1.0).cpu()
    return TF.to_pil_image(out)

@torch.no_grad()
def test_mode_A_clean_to_noisy(clean_path, ckpt_path, out_dir="test_outputs_drunet", sigma=25.0, seed=0):
    os.makedirs(out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = DRUNetSigmaMap(in_nc=4, out_nc=3, nc=(64,128,256,512), nb=4).to(device).eval()
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model"], strict=True)

    clean_pil = Image.open(clean_path).convert("RGB")
    clean = TF.to_tensor(clean_pil).unsqueeze(0)  # (1,3,H,W)

    g = torch.Generator().manual_seed(seed)
    noise = torch.randn(clean.shape, generator=g) * (sigma / 255.0)
    noisy = clean + noise  # PAS de clamp ici non plus (cohérent avec train)

    sigma_map = torch.full((1, 1, clean.shape[2], clean.shape[3]), sigma / 255.0)
    inp = torch.cat([noisy, sigma_map], dim=1).to(device)
    den = model(inp).clamp(0.0, 1.0).cpu()

    noisy_clamped = noisy.clamp(0.0, 1.0)  # seulement pour sauvegarde/PSNR
    psnr_noisy = psnr_torch(noisy_clamped, clean)
    psnr_den = psnr_torch(den, clean)

    TF.to_pil_image(clean.squeeze(0)).save(os.path.join(out_dir, "clean.png"))
    TF.to_pil_image(noisy_clamped.squeeze(0)).save(os.path.join(out_dir, f"noisy_sigma{int(sigma)}.png"))
    TF.to_pil_image(den.squeeze(0)).save(os.path.join(out_dir, f"denoised_sigma{int(sigma)}.png"))

    print("Saved to:", out_dir)
    print(f"PSNR noisy   : {psnr_noisy:.2f} dB")
    print(f"PSNR denoised: {psnr_den:.2f} dB")


In [27]:
test_mode_A_clean_to_noisy(
        clean_path=r"./BSDS300/images/test/102061.jpg",
        ckpt_path=r"./weights_drunet_sigmap/drunet_sigmap_final.pth",
        out_dir="test_outputs_denoise_drunet",
        sigma=70.0,
        seed=0
    )

C:\Users\barra\AppData\Local\Temp\ipykernel_23760\375610018.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location=device)


Saved to: test_outputs_denoise_drunet
PSNR noisy   : 12.59 dB
PSNR denoised: 27.16 dB


# Deblurrin